# Metop (EPS) / ASCAT (Advanced Scatterometer)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=metop/metop-ascat). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L2

Level 2: ocean surface wind vectors on 25 km and 12.5 km grids (coastal products), with quality flags, in BUFR/netCDF; Level 1B backscatter (SZR/SZF) is distributed separately.

Downloads one ASCAT 25 km wind product from the EUMETSAT Data Store (free account and consumer key/secret) and plots wind speed along the swath.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:METOP:OAS025")

products = list(collection.search(dtstart=dt.datetime(2024, 6, 1, 12, 0), dtend=dt.datetime(2024, 6, 1, 14, 0)))
print(len(products), "ASCAT 25 km wind products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# a few MB per orbit
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

import xarray as xr

files = sorted(out.rglob("*.nc"))
print(files[:3])
if files:
    try:
        ds = xr.open_dataset(files[0], engine="h5netcdf")
        print(ds)
    except Exception as exc:                      # some products need group="..." or another engine
        print("Could not open directly:", exc)
else:
    print("No .nc file: the product is in EPS native / BUFR / HDF5 format - read it with satpy, eccodes or the EUMETSAT tools")

if files and "wind_speed" in ds:
    import matplotlib.pyplot as plt
    plt.scatter(ds["lon"].values.ravel(), ds["lat"].values.ravel(), c=ds["wind_speed"].values.ravel(), s=3, cmap="viridis")
    plt.colorbar(label="wind speed (m/s)")
    plt.show()